In [ ]:
# =========================================================================
#
#       COPD Full Diagnostic Report Pipeline (Definitive Version)
#
# This script uses the final, user-specified configuration where the
# Stage 1 vs. 2 model uses N_MELS=128, and all other models use
# N_MELS=20. It provides a detailed, vote-counted report.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings 

warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
PATIENT_ID_TO_ANALYZE = "221"
MODELS_BASE_PATH = "./models/"

# --- FINAL Configuration for Expert Models ---
MODEL_PIPELINE = [
    {
        "name": "Healthy(0) vs. Early Stage(1)",
        "model_path": os.path.join(MODELS_BASE_PATH, "testing_0_1.keras"),
        "mean_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_0_1_mel20_mean.npy"),
        "std_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_0_1_mel20_std.npy"),
        "n_mels": 20,
        "labels": {0: 'COPD0', 1: 'COPD1'},
        "has_custom_layer": True
    },
    {
        "name": "Stage 1 vs. Stage 2",
        "model_path": os.path.join(MODELS_BASE_PATH, "testing_1_2.keras"),
        "mean_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_1_2_mel20_mean.npy"),
        "std_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_1_2_mel20_std.npy"),
        "n_mels": 128, # CRUCIAL: Set to 128 to match the data inside the stats file
        "labels": {0: 'COPD1', 1: 'COPD2'},
        "has_custom_layer": True
    },
    {
        "name": "Stage 2 vs. Stage 3",
        "model_path": os.path.join(MODELS_BASE_PATH, "testing_2_3.keras"),
        "mean_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_2_3_mel20_mean.npy"),
        "std_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_2_3_mel20_std.npy"),
        "n_mels": 20,
        "labels": {0: 'COPD2', 1: 'COPD3'},
        "has_custom_layer": True
    },
    {
        "name": "Stage 3 vs. Stage 4",
        "model_path": os.path.join(MODELS_BASE_PATH, "testing_3_4.keras"),
        "mean_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_3_4_mel20_mean.npy"),
        "std_path": os.path.join(MODELS_BASE_PATH, "logmel_testing_3_4_mel20_std.npy"),
        "n_mels": 20,
        "labels": {0: 'COPD3', 1: 'COPD4'},
        "has_custom_layer": True
    }
]
MAX_LEN = 150

# --- Step 2: Define Custom Layer ---
class GroupNormalization(Layer):
    """Needed for loading models trained with this custom layer."""
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs)
        self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1];self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones');self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros')
    def call(self, inputs):
        s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta

# --- Step 3: Helper & Prediction Functions ---
def extract_and_process_spectrogram(path, n_mels, max_len, mean, std):
    """A robust, self-contained function to process one file with dynamic n_mels."""
    y, sr = librosa.load(path, sr=None)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < max_len: log_mel = np.pad(log_mel, ((0,0), (0, max_len - log_mel.shape[1])))
    else: log_mel = log_mel[:, :max_len]
    normalized = (log_mel - mean) / (std + 1e-6)
    return normalized[np.newaxis, ..., np.newaxis]

def predict_patient_with_single_model(patient_id, audio_dir, config):
    """Loads a model and gets a majority vote, returning detailed results."""
    error_result = {"prediction": "ERROR", "vote_details": "N/A"}
    try:
        custom_objects = {"GroupNormalization": GroupNormalization} if config['has_custom_layer'] else None
        model = load_model(config['model_path'], custom_objects=custom_objects, compile=False)
        training_mean, training_std = np.load(config['mean_path']), np.load(config['std_path'])
    except Exception as e:
        error_result["prediction"] = f"ERROR loading files for '{config['name']}'"
        error_result["vote_details"] = str(e)
        return error_result
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    if not patient_files: error_result["prediction"] = "No Audio Files Found"; return error_result
    
    predictions_idx = []
    for fpath in patient_files:
        try:
            log_mel_final = extract_and_process_spectrogram(fpath, config['n_mels'], MAX_LEN, training_mean, training_std)
            prob = model.predict(log_mel_final, verbose=0)[0][0]
            predictions_idx.append(1 if prob >= 0.5 else 0)
        except Exception as e:
            error_msg = str(e)
            if 'operands could not be broadcast' in error_msg:
                 error_msg += f" (Hint: Check if n_mels={config['n_mels']} matches the loaded stats file)"
            print(f"  -> Warning: Could not process {os.path.basename(fpath)}: {error_msg}")

    if not predictions_idx: error_result["prediction"] = "Processing Error"; return error_result
    
    majority_vote_idx = max(set(predictions_idx), key=predictions_idx.count)
    vote_details_str = f"({config['labels'][0]}: {predictions_idx.count(0)}, {config['labels'][1]}: {predictions_idx.count(1)})"
    
    return {"prediction": config['labels'][majority_vote_idx], "vote_details": vote_details_str}

# --- Step 4: The Main Diagnostic Report Pipeline ---
def run_diagnostic_report(patient_id, audio_dir):
    """Orchestrates the process and prints the final report with vote counts."""
    print("="*60)
    print(f"--- 🩺 Starting Full Diagnostic Report for Patient ID: {patient_id} ---")
    
    report_data = []
    
    for stage_config in MODEL_PIPELINE:
        print(f"\n--- Consulting Expert Model: {stage_config['name']} ---")
        result_dict = predict_patient_with_single_model(patient_id, audio_dir, stage_config)
        
        print(f"  -> Verdict: {result_dict['prediction']} | Vote Distribution: {result_dict['vote_details']}")
        
        report_data.append({
            "Model": stage_config['name'],
            "Prediction": result_dict['prediction'],
            "Vote Details": result_dict['vote_details']
        })
            
    print("\n" + "="*80)
    print(f"--- 📋 COMPREHENSIVE DIAGNOSTIC REPORT for Patient {patient_id} ---")
    
    report_df = pd.DataFrame(report_data)
    print(report_df.to_string(index=False))
    
    print("\nThis report shows the majority vote and vote distribution from each independent model.")
    print("="*80)

# --- Run the main analysis function ---
if __name__ == "__main__":
    run_diagnostic_report(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

2025-07-11 11:35:10.164968: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-11 11:35:10.171977: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-11 11:35:10.192991: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752213910.228505  255356 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752213910.238435  255356 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752213910.266714  255356 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

--- 🩺 Starting Full Diagnostic Report for Patient ID: 221 ---

--- Consulting Expert Model: Healthy(0) vs. Early Stage(1) ---


2025-07-11 11:35:13.943898: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


  -> Verdict: COPD0 | Vote Distribution: (COPD0: 12, COPD1: 0)

--- Consulting Expert Model: Stage 1 vs. Stage 2 ---


2025-07-11 11:35:18.456847: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 38797312 exceeds 10% of free system memory.
2025-07-11 11:35:18.467523: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 38797312 exceeds 10% of free system memory.
2025-07-11 11:35:18.473905: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 38797312 exceeds 10% of free system memory.
2025-07-11 11:35:18.764701: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 38797312 exceeds 10% of free system memory.


  -> Verdict: COPD2 | Vote Distribution: (COPD1: 0, COPD2: 12)

--- Consulting Expert Model: Stage 2 vs. Stage 3 ---
  -> Verdict: COPD2 | Vote Distribution: (COPD2: 12, COPD3: 0)

--- Consulting Expert Model: Stage 3 vs. Stage 4 ---
  -> Verdict: COPD3 | Vote Distribution: (COPD3: 11, COPD4: 1)

--- 📋 COMPREHENSIVE DIAGNOSTIC REPORT for Patient 221 ---
                        Model Prediction          Vote Details
Healthy(0) vs. Early Stage(1)      COPD0 (COPD0: 12, COPD1: 0)
          Stage 1 vs. Stage 2      COPD2 (COPD1: 0, COPD2: 12)
          Stage 2 vs. Stage 3      COPD2 (COPD2: 12, COPD3: 0)
          Stage 3 vs. Stage 4      COPD3 (COPD3: 11, COPD4: 1)

This report shows the majority vote and vote distribution from each independent model.


In [2]:
# =========================================================================
#
#       Combined Diagnostic Script: Multi-Class Classifier + Expert Pipeline
#
# This script performs a two-part analysis for a given patient:
#   1. Runs a primary multi-class classifier for an initial diagnosis.
#   2. Runs a secondary "chain of experts" pipeline for a detailed report.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENT_ID_TO_ANALYZE = "221"

# --- Configuration for Part 1: Primary Multi-Class Classifier ---
CLASSIFIER_MODEL_CONFIG = {
    "name": "Primary Multi-Class Classifier",
    "model_path": os.path.join(MODELS_BASE_PATH, "best_copd_model_finetuned.h5"),
    "mean_path": os.path.join(MODELS_BASE_PATH, "mfcc_training_mean.npy"),
    "std_path": os.path.join(MODELS_BASE_PATH, "mfcc_training_std.npy"),
    "labels": ['COPD0', 'COPD1', 'COPD2', 'COPD3', 'COPD4'],
    "n_mfcc": 20,
    "max_len": 150
}

# --- Configuration for Part 2: Expert Pipeline ---
EXPERT_PIPELINE_CONFIG = [
    {"name":"Healthy(0) vs. Early Stage(1)","model_path":os.path.join(MODELS_BASE_PATH,"testing_0_1.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD0',1:'COPD1'},"has_custom_layer":True},
    {"name":"Stage 1 vs. Stage 2","model_path":os.path.join(MODELS_BASE_PATH,"testing_1_2.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_std.npy"),"n_mels":128,"labels":{0:'COPD1',1:'COPD2'},"has_custom_layer":False}, # Corrected based on your files
    {"name":"Stage 2 vs. Stage 3","model_path":os.path.join(MODELS_BASE_PATH,"testing_2_3.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD2',1:'COPD3'},"has_custom_layer":True},
    {"name":"Stage 3 vs. Stage 4","model_path":os.path.join(MODELS_BASE_PATH,"testing_3_4.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD3',1:'COPD4'},"has_custom_layer":True}
]

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    """Custom layer needed for loading some expert models."""
    def __init__(self, groups=4, epsilon=1e-5, **kwargs): super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape): dim=input_shape[-1];self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones');self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros')
    def call(self, inputs): s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta

def run_primary_classifier(patient_id, audio_dir, config):
    """Runs the primary multi-class classifier on a patient's recordings."""
    print("="*60)
    print(f"--- 1. Running Primary Multi-Class Classifier for Patient {patient_id} ---")
    try:
        model = load_model(config['model_path'])
        mean, std = np.load(config['mean_path']), np.load(config['std_path'])
    except Exception as e: print(f"❌ ERROR loading classifier files: {e}"); return
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)) and f.endswith('.wav')]
    if not patient_files: print(f"❌ No .wav files found."); return
    print(f"\nFound {len(patient_files)} files. Analyzing with primary classifier (MFCC based)...")
    predictions = []
    for fpath in sorted(patient_files):
        try:
            y, sr = librosa.load(fpath, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=config['n_mfcc'])
            if mfcc.shape[1] < config['max_len']: mfcc = np.pad(mfcc, ((0,0), (0, config['max_len'] - mfcc.shape[1])))
            else: mfcc = mfcc[:, :config['max_len']]
            mfcc_norm = (mfcc - mean) / (std + 1e-6); mfcc_final = mfcc_norm[np.newaxis,...,np.newaxis]
            probs = model.predict(mfcc_final, verbose=0)[0]
            pred_idx = np.argmax(probs); predictions.append(config['labels'][pred_idx])
            print(f"  -> {os.path.basename(fpath):<35} | Predicted: {config['labels'][pred_idx]:<6} | Confidence: {probs[pred_idx]*100:.2f}%")
        except Exception as e: print(f"  -> ERROR processing file {os.path.basename(fpath)}: {e}")
    if predictions:
        final_verdict = max(set(predictions), key=predictions.count)
        print("\n" + "-"*50 + f"\nPrimary Classifier Verdict for Patient {patient_id}: >> {final_verdict} <<\n" + "-"*50)

def run_expert_pipeline_report(patient_id, audio_dir):
    """Runs the secondary chain of expert models."""
    print("\n\n" + "="*60)
    print(f"--- 2. Running Detailed Expert Model Pipeline for Patient ID: {patient_id} ---")
    report_data = []
    for config in EXPERT_PIPELINE_CONFIG:
        print(f"\n--- Consulting Expert Model: {config['name']} ---")
        verdict = predict_with_expert_model(patient_id, audio_dir, config)
        print(f"  -> Verdict: {verdict['prediction']} | Vote Distribution: {verdict['vote_details']}")
        report_data.append({"Model": config['name'], "Prediction": verdict['prediction'], "Vote Details": verdict['vote_details']})
    print("\n" + "="*80)
    print(f"--- 📋 COMPREHENSIVE EXPERT REPORT for Patient {patient_id} ---\n" + "="*80)
    print(pd.DataFrame(report_data).to_string(index=False))

def predict_with_expert_model(patient_id, audio_dir, config):
    """The prediction logic for a single expert model."""
    error_result = {"prediction": "ERROR", "vote_details": "N/A"}
    try:
        custom_objects = {"GroupNormalization": GroupNormalization} if config.get('has_custom_layer', False) else None
        model = load_model(config['model_path'], custom_objects=custom_objects, compile=False)
        mean, std = np.load(config['mean_path']), np.load(config['std_path'])
    except Exception as e:
        error_result["prediction"] = f"ERROR loading '{os.path.basename(config['model_path'])}'"; error_result["vote_details"] = str(e); return error_result
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]
    if not patient_files: error_result["prediction"] = "No Audio Files Found"; return error_result
    predictions = []
    for fpath in patient_files:
        try:
            y, sr = librosa.load(fpath, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=config['n_mels']); log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < EXPERT_PIPELINE_CONFIG[0].get('max_len', 150): log_mel = np.pad(log_mel, ((0,0), (0, EXPERT_PIPELINE_CONFIG[0].get('max_len', 150) - log_mel.shape[1])))
            else: log_mel = log_mel[:, :EXPERT_PIPELINE_CONFIG[0].get('max_len', 150)]
            log_mel_norm = (log_mel - mean) / (std + 1e-6); log_mel_final = log_mel_norm[np.newaxis,...,np.newaxis]
            prob = model.predict(log_mel_final, verbose=0)[0][0]
            predictions.append(1 if prob >= 0.5 else 0)
        except Exception as e: print(f"  -> Warning: {e}")
    if not predictions: error_result["prediction"] = "Processing Error"; return error_result
    majority_vote = max(set(predictions), key=predictions.count)
    vote_details = f"({config['labels'][0]}: {predictions.count(0)}, {config['labels'][1]}: {predictions.count(1)})"
    return {"prediction": config['labels'][majority_vote], "vote_details": vote_details}

# --- Main execution block ---
if __name__ == "__main__":
    run_primary_classifier(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR, CLASSIFIER_MODEL_CONFIG)
    run_expert_pipeline_report(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- 1. Running Primary Multi-Class Classifier for Patient 221 ---



Found 12 files. Analyzing with primary classifier (MFCC based)...
  -> ERROR processing file 221_2b1_Al_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b1_Ar_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b1_Lr_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b1_Pl_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b2_Al_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b2_Ar_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b2_Lr_mc_LittC2SE.wav: operands could not be broadcast together with shapes (20,150) (128,150) 
  -> ERROR processing file 221_2b2_Pl

2025-07-11 11:35:29.237699: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 38797312 exceeds 10% of free system memory.


  -> Verdict: COPD2 | Vote Distribution: (COPD1: 0, COPD2: 12)

--- Consulting Expert Model: Stage 2 vs. Stage 3 ---
  -> Verdict: COPD2 | Vote Distribution: (COPD2: 12, COPD3: 0)

--- Consulting Expert Model: Stage 3 vs. Stage 4 ---
  -> Verdict: COPD3 | Vote Distribution: (COPD3: 11, COPD4: 1)

--- 📋 COMPREHENSIVE EXPERT REPORT for Patient 221 ---
                        Model Prediction          Vote Details
Healthy(0) vs. Early Stage(1)      COPD0 (COPD0: 12, COPD1: 0)
          Stage 1 vs. Stage 2      COPD2 (COPD1: 0, COPD2: 12)
          Stage 2 vs. Stage 3      COPD2 (COPD2: 12, COPD3: 0)
          Stage 3 vs. Stage 4      COPD3 (COPD3: 11, COPD4: 1)


In [1]:
# =========================================================================
#
#       Combined Diagnostic Script (Standardized on N_MELS=20)
#
# This definitive version assumes ALL models (primary classifier and all
# four experts) have been trained with a consistent n_mels=20 configuration.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
tf.get_logger().setLevel('ERROR')

# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENTS_TO_ANALYZE = ["104", "106", "107", "109", "223"]

# --- Global Feature Parameters (Consistent for all models) ---
MAX_LEN = 150
N_MELS_LOGMEL = 20 # Standardized for all expert models
N_MFCC = 20 # For the primary classifier

# --- Configuration for Part 1: Primary Multi-Class Classifier (MFCC-based) ---
CLASSIFIER_MODEL_CONFIG = {
    "name": "PrimaryClassifier",
    "model_path": os.path.join(MODELS_BASE_PATH,"best_copd_model_finetuned.h5"),
    "mean_path": os.path.join(MODELS_BASE_PATH,"mfcc_training_mean.npy"),
    "std_path": os.path.join(MODELS_BASE_PATH,"mfcc_training_std.npy"),
    "labels": ['COPD0', 'COPD1', 'COPD2', 'COPD3', 'COPD4'],
    "has_custom_layer": False # Assuming standard Sequential model
}

# --- Configuration for Part 2: Expert Pipeline (Log-Mel based, all N_MELS=20) ---
EXPERT_PIPELINE_CONFIG = [
    {"name":"Expert 0vs1","model_path":os.path.join(MODELS_BASE_PATH,"testing_0_1.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_std.npy"),"labels":{0:'COPD0',1:'COPD1'},"has_custom_layer":True},
    {"name":"Expert 1vs2","model_path":os.path.join(MODELS_BASE_PATH,"testing_1_2.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_std.npy"),"labels":{0:'COPD1',1:'COPD2'},"has_custom_layer":True},
    {"name":"Expert 2vs3","model_path":os.path.join(MODELS_BASE_PATH,"testing_2_3.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_std.npy"),"labels":{0:'COPD2',1:'COPD3'},"has_custom_layer":True},
    {"name":"Expert 3vs4","model_path":os.path.join(MODELS_BASE_PATH,"testing_3_4.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_std.npy"),"labels":{0:'COPD3',1:'COPD4'},"has_custom_layer":True}
]

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs): super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon=groups,epsilon
    def build(self, input_shape): dim=input_shape[-1];self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones');self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros')
    def call(self, inputs): s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta

def get_patient_files(patient_id, audio_dir):
    return [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]

def predict_with_primary_classifier(patient_files, resources):
    """Predicts using the main multi-class classifier (MFCC based)."""
    if not patient_files: return "No Audio Found"
    predictions = []
    for fpath in patient_files:
        try:
            y, sr = librosa.load(fpath, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
            if mfcc.shape[1] < MAX_LEN: mfcc = np.pad(mfcc, ((0,0), (0, MAX_LEN - mfcc.shape[1])))
            else: mfcc = mfcc[:, :MAX_LEN]
            mfcc_norm = (mfcc - resources['mean']) / (resources['std'] + 1e-6); mfcc_final = mfcc_norm[np.newaxis,...,np.newaxis]
            probs = resources['model'].predict(mfcc_final, verbose=0)[0]
            predictions.append(np.argmax(probs))
        except Exception: continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions), key=predictions.count)]

def predict_with_expert_model(patient_files, resources):
    """Predicts using a single binary expert model (Log-Mel based)."""
    if not patient_files: return "No Audio Found"
    predictions = []
    for fpath in patient_files:
        try:
            y, sr = librosa.load(fpath, sr=None)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS_LOGMEL) # Use the global Log-Mel setting
            log_mel = librosa.power_to_db(mel)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            norm = (log_mel - resources['mean']) / (resources['std'] + 1e-6); final = norm[np.newaxis,...,np.newaxis]
            prob = resources['model'].predict(final, verbose=0)[0][0]
            predictions.append(1 if prob >= 0.5 else 0)
        except Exception: continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions), key=predictions.count)]


def main(patient_list, audio_dir):
    """Orchestrates the loading of models and the analysis for a list of patients."""
    # Pre-load all resources to prevent retracing warnings
    print("--- Pre-loading all models and resources... ---")
    loaded_resources = {}
    custom_objects = {"GroupNormalization": GroupNormalization}
    for config in [CLASSIFIER_MODEL_CONFIG] + EXPERT_PIPELINE_CONFIG:
        try:
            is_expert = 'n_mels' in config
            model_type = "Expert" if is_expert else "Classifier"
            print(f"  Loading {model_type}: {config['name']}...")
            model = load_model(config['model_path'], custom_objects=custom_objects if config.get('has_custom_layer') else None, compile=False)
            mean, std = np.load(config['mean_path']), np.load(config['std_path'])
            loaded_resources[config['name']] = {**config, 'model': model, 'mean': mean, 'std': std}
        except Exception as e:
            print(f"❌ Critical Error loading '{config['name']}'. Check paths. Error: {e}")
            loaded_resources[config['name']] = None

    print("✅ All models loaded (or marked as failed).")
    
    # Batch Processing
    print(f"\n--- 🩺 Starting Batch Diagnostic Run for {len(patient_list)} Patients ---")
    final_results = []
    for patient_id in patient_list:
        print(f"\n--- Analyzing Patient ID: {patient_id} ---")
        patient_report = {"Patient ID": patient_id}
        patient_files = get_patient_files(patient_id, audio_dir)

        # 1. Run the primary classifier
        primary_res = loaded_resources.get(CLASSIFIER_MODEL_CONFIG['name'])
        patient_report["Classified Stage"] = predict_with_primary_classifier(patient_files, primary_res) if primary_res else "Model Load Failed"
        
        # 2. Run the expert model pipeline
        for expert_config in EXPERT_PIPELINE_CONFIG:
            expert_res = loaded_resources.get(expert_config['name'])
            col_name = expert_config['name'].replace("Early Stage", "Stage").replace("Healthy", "H").replace("Expert ", "")
            patient_report[col_name] = predict_with_expert_model(patient_files, expert_res) if expert_res else "Model Load Failed"
            
        final_results.append(patient_report)

    # Final Summary Table
    print("\n\n" + "="*90 + "\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n" + "="*90)
    if final_results:
        results_df = pd.DataFrame(final_results)
        print(results_df.to_string(index=False))
    else:
        print("No results were generated.")
    print("="*90)

if __name__ == "__main__":
    main(set(PATIENTS_TO_ANALYZE), NEW_AUDIO_DIR)

2025-07-13 19:56:04.159782: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:56:04.166821: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:56:04.189061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752416764.233530    4862 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752416764.244929    4862 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752416764.274377    4862 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

--- Pre-loading all models and resources... ---
  Loading Classifier: PrimaryClassifier...


2025-07-13 19:56:08.704088: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


  Loading Classifier: Expert 0vs1...
  Loading Classifier: Expert 1vs2...
  Loading Classifier: Expert 2vs3...
  Loading Classifier: Expert 3vs4...
✅ All models loaded (or marked as failed).

--- 🩺 Starting Batch Diagnostic Run for 5 Patients ---

--- Analyzing Patient ID: 107 ---

--- Analyzing Patient ID: 106 ---

--- Analyzing Patient ID: 223 ---

--- Analyzing Patient ID: 104 ---

--- Analyzing Patient ID: 109 ---


--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---
Patient ID Classified Stage  0vs1  1vs2  2vs3  3vs4
       107            COPD3 COPD0 COPD1 COPD3 COPD3
       106            COPD3 COPD0 COPD2 COPD2 COPD3
       223            COPD4 COPD0 COPD1 COPD3 COPD3
       104            COPD4 COPD1 COPD2 COPD3 COPD3
       109            COPD4 COPD1 COPD2 COPD2 COPD3


try 2 with classifier model also train with only 20 mels

In [ ]:
# # =========================================================================
# #
# #       COPD Final Diagnostic Pipeline (Definitive, Corrected Version)
# #
# # This script contains the final fix for the TypeError in GroupNormalization.
# #
# # =========================================================================

# import os
# import librosa
# import numpy as np
# import pandas as pd
# import tensorflow as tf
# from tensorflow.keras.models import load_model
# from tensorflow.keras.layers import Layer
# import warnings

# warnings.filterwarnings('ignore', category=UserWarning)
# tf.get_logger().setLevel('ERROR')

# # --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
# PATIENTS_TO_ANALYZE = ["104", "106", "107", "109", "223"]

# N_MELS, MAX_LEN, N_MFCC = 20, 150, 20

# ALL_MODELS_CONFIG = [
#     {"name":"Primary Classifier", "model_path":"best_copd_model_finetuned.h5", "mean_path":"mfcc_training_mean.npy", "std_path":"mfcc_training_std.npy", "labels":['COPD0','COPD1','COPD2','COPD3','COPD4'], "feature_type":"mfcc", "has_custom_layer":False},
#     {"name":"Expert 0vs1", "model_path":"testing_0_1.keras", "mean_path":"logmel_testing_0_1_mel20_mean.npy", "std_path":"logmel_testing_0_1_mel20_std.npy", "labels":{0:'COPD0',1:'COPD1'}, "feature_type":"logmel", "has_custom_layer":True},
#     {"name":"Expert 1vs2", "model_path":"testing_1_2.keras", "mean_path":"logmel_testing_1_2_mel20_mean.npy", "std_path":"logmel_testing_1_2_mel20_std.npy", "labels":{0:'COPD1',1:'COPD2'}, "feature_type":"logmel", "has_custom_layer":True},
#     {"name":"Expert 2vs3", "model_path":"testing_2_3.keras", "mean_path":"logmel_testing_2_3_mel20_mean.npy", "std_path":"logmel_testing_2_3_mel20_std.npy", "labels":{0:'COPD2',1:'COPD3'}, "feature_type":"logmel", "has_custom_layer":True},
#     {"name":"Expert 3vs4", "model_path":"testing_3_4.keras", "mean_path":"logmel_testing_3_4_mel20_mean.npy", "std_path":"logmel_testing_3_4_mel20_std.npy", "labels":{0:'COPD3',1:'COPD4'}, "feature_type":"logmel", "has_custom_layer":True}
# ]

# # --- Helper Layers & Functions ---
# # --- CORRECTED GroupNormalization Layer ---
# class GroupNormalization(Layer):
#     """Custom Group Normalization layer with correct arguments."""
#     def __init__(self, groups=4, epsilon=1e-5, **kwargs):
#         super(GroupNormalization, self).__init__(**kwargs)
#         self.groups = groups
#         self.epsilon = epsilon
        
#     def build(self, input_shape):
#         dim = input_shape[-1]
#         # Using the full, correct keyword arguments: 'name', 'shape', 'initializer'
#         self.gamma = self.add_weight(name='gamma', shape=(1,1,1,dim), initializer='ones')
#         self.beta = self.add_weight(name='beta', shape=(1,1,1,dim), initializer='zeros')

#     def call(self, inputs):
#         input_shape = tf.shape(inputs)
#         N, H, W, C = input_shape[0], input_shape[1], input_shape[2], input_shape[3]
#         group_size = C // self.groups
#         reshaped = tf.reshape(inputs, [N, H, W, self.groups, group_size])
#         mean, var = tf.nn.moments(reshaped, [1, 2, 4], keepdims=True)
#         normalized = (reshaped - mean) / tf.sqrt(var + self.epsilon)
#         return tf.reshape(normalized, input_shape) * self.gamma + self.beta


# def get_patient_files(patient_id, audio_dir):
#     return [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]


# def get_prediction(patient_files, resources, feature_type):
#     """Unified prediction function for both MFCC and Log-Mel models."""
#     if not patient_files: return "No Audio Found"
#     if resources is None: return "Model Load Failed"
    
#     predictions = []
#     is_multiclass = isinstance(resources['labels'], list)

#     for fpath in patient_files:
#         try:
#             if feature_type == 'mfcc':
#                 y, sr = librosa.load(fpath, sr=16000)
#                 feature = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
#             else:
#                 y, sr = librosa.load(fpath, sr=None)
#                 mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
#                 feature = librosa.power_to_db(mel_spec)

#             if feature.shape[1] < MAX_LEN: feature = np.pad(feature, ((0,0), (0, MAX_LEN - feature.shape[1])))
#             else: feature = feature[:, :MAX_LEN]
            
#             normalized_feature = (feature - resources['mean']) / (resources['std'] + 1e-6)
#             final_input = normalized_feature[np.newaxis, ..., np.newaxis]
#             probs = resources['model'].predict(final_input, verbose=0)[0]
            
#             if is_multiclass:
#                 predictions.append(np.argmax(probs))
#             else:
#                 predictions.append(1 if probs[0] >= 0.5 else 0)
#         except Exception as e:
#             continue
            
#     if not predictions: return "Processing Error"
    
#     majority_vote_idx = max(set(predictions), key=predictions.count)
#     return resources['labels'][majority_vote_idx]


# # --- Main Pipeline ---
# def main(patient_list, audio_dir):
#     """Orchestrates model loading and batch analysis."""
#     print("--- Pre-loading all models and resources... ---")
#     loaded_resources = {}
#     custom_objects = {"GroupNormalization": GroupNormalization}
    
#     for config in ALL_MODELS_CONFIG:
#         try:
#             full_model_path = os.path.join(MODELS_BASE_PATH, config['model_path'])
#             full_mean_path = os.path.join(MODELS_BASE_PATH, config['mean_path'])
#             full_std_path = os.path.join(MODELS_BASE_PATH, config['std_path'])

#             model = load_model(full_model_path, custom_objects=custom_objects if config.get('has_custom_layer') else None, compile=False)
#             mean, std = np.load(full_mean_path), np.load(full_std_path)
#             loaded_resources[config['name']] = {**config, 'model': model, 'mean': mean, 'std': std}
#             print(f"  ✅ Loaded: {config['name']}")
#         except Exception as e:
#             print(f"  ❌ FAILED to load model '{config['name']}': {e}")
#             loaded_resources[config['name']] = None
            
#     print("--- Model loading complete. ---\n")
    
#     final_results = []
#     for patient_id in patient_list:
#         print(f"--- Analyzing Patient ID: {patient_id} ---")
#         patient_report = {"Patient ID": patient_id}
#         patient_files = get_patient_files(patient_id, audio_dir)
        
#         for config in ALL_MODELS_CONFIG:
#             res = loaded_resources.get(config['name'])
#             col_name = "Classified Stage" if config['feature_type'] == 'mfcc' else config['name'].replace("Expert ", "")
#             patient_report[col_name] = get_prediction(patient_files, res, config['feature_type'])
            
#         final_results.append(patient_report)
    
#     print("\n\n" + "="*90 + "\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n" + "="*90)
#     results_df = pd.DataFrame(final_results)
#     print(results_df.to_string(index=False))
#     print("="*90)


# if __name__ == "__main__":
#     main(sorted(list(set(PATIENTS_TO_ANALYZE))), NEW_AUDIO_DIR)

--- Pre-loading all models and resources... ---
  ✅ Loaded: Primary Classifier
  ✅ Loaded: Expert 0vs1
  ✅ Loaded: Expert 1vs2
  ✅ Loaded: Expert 2vs3
  ✅ Loaded: Expert 3vs4
--- Model loading complete. ---

--- Analyzing Patient ID: 104 ---
--- Analyzing Patient ID: 106 ---
--- Analyzing Patient ID: 107 ---
--- Analyzing Patient ID: 109 ---
--- Analyzing Patient ID: 223 ---


--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---
Patient ID Classified Stage  0vs1             1vs2  2vs3  3vs4
       104 Processing Error COPD1 Processing Error COPD3 COPD3
       106 Processing Error COPD0 Processing Error COPD2 COPD3
       107 Processing Error COPD0 Processing Error COPD3 COPD4
       109 Processing Error COPD1 Processing Error COPD2 COPD3
       223 Processing Error COPD0 Processing Error COPD2 COPD3


In [3]:
# =========================================================================
#
#       COPD Batch Diagnostic Pipeline (Optimized Version)
#
# This version pre-loads all models at the start to prevent inefficient
# re-tracing and speed up the prediction process significantly.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
# This will suppress the harmless "Compiled metrics" warning
tf.get_logger().setLevel('ERROR')


# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENTS_TO_ANALYZE = [
    "104", "106", "107", "109", "110", "112", "113", "114",
    "117", "118", "120", "124", "128", "130", "132", "133",
    "134", "138", "139", "141", "142", "145", "146", "147",
    "151", "154", "155", "156", "157", "158", "160", "162",
    "163", "166", "170", "172", "174", "175", "176", "177",
    "178", "180", "181", "185", "186", "189", "192", "193",
    "195", "198", "199", "200", "203", "204", "205", "207",
    "211", "212", "213", "218", "220", "221", "222", "223"
]

CLASSIFIER_MODEL_CONFIG = { "name": "PrimaryClassifier", "model_path": os.path.join(MODELS_BASE_PATH,"best_copd_model_finetuned.h5"), "mean_path": os.path.join(MODELS_BASE_PATH,"mfcc_training_mean.npy"), "std_path": os.path.join(MODELS_BASE_PATH,"mfcc_training_std.npy"), "labels": ['COPD0', 'COPD1', 'COPD2', 'COPD3', 'COPD4'], "n_features": 20, "max_len": 150, "has_custom_layer": False}
EXPERT_PIPELINE_CONFIG = [
    {"name":"Expert 0vs1","model_path":os.path.join(MODELS_BASE_PATH,"testing_0_1.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD0',1:'COPD1'},"has_custom_layer":True},
    {"name":"Expert 1vs2","model_path":os.path.join(MODELS_BASE_PATH,"testing_1_2.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD1',1:'COPD2'},"has_custom_layer":True},
    {"name":"Expert 2vs3","model_path":os.path.join(MODELS_BASE_PATH,"testing_2_3.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD2',1:'COPD3'},"has_custom_layer":True},
    {"name":"Expert 3vs4","model_path":os.path.join(MODELS_BASE_PATH,"testing_3_4.keras"),"mean_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_mean.npy"),"std_path":os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_std.npy"),"n_mels":20,"labels":{0:'COPD3',1:'COPD4'},"has_custom_layer":True}
]

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs): super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon=groups,epsilon
    def build(self, input_shape): dim=input_shape[-1];self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones');self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros')
    def call(self, inputs): s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta

def get_patient_files(patient_id, audio_dir):
    return [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]

# --- OPTIMIZED Prediction Functions (they now take loaded resources as arguments) ---
def predict_with_primary_classifier(patient_files, resources):
    if not patient_files: return "No Audio"
    predictions = []
    for fpath in patient_files:
        try:
            y, sr = librosa.load(fpath, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=resources['n_features'])
            if mfcc.shape[1] < resources['max_len']: mfcc = np.pad(mfcc, ((0,0), (0, resources['max_len'] - mfcc.shape[1])))
            else: mfcc = mfcc[:, :resources['max_len']]
            mfcc_norm = (mfcc - resources['mean']) / (resources['std'] + 1e-6); mfcc_final = mfcc_norm[np.newaxis,...,np.newaxis]
            probs = resources['model'].predict(mfcc_final, verbose=0)[0]
            predictions.append(np.argmax(probs))
        except Exception: continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions), key=predictions.count)]

def predict_with_expert_model(patient_files, resources):
    if not patient_files: return "No Audio"
    predictions = []
    for fpath in patient_files:
        try:
            y, sr = librosa.load(fpath, sr=None)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=resources['n_mels']); log_mel = librosa.power_to_db(mel)
            if log_mel.shape[1] < resources.get('max_len', 150): log_mel = np.pad(log_mel, ((0,0), (0, resources.get('max_len', 150) - log_mel.shape[1])))
            else: log_mel = log_mel[:, :resources.get('max_len', 150)]
            norm = (log_mel - resources['mean']) / (resources['std'] + 1e-6); final = norm[np.newaxis,...,np.newaxis]
            prob = resources['model'].predict(final, verbose=0)[0][0]
            predictions.append(1 if prob >= 0.5 else 0)
        except Exception as e:
            if 'operands could not be broadcast' in str(e): return f"Shape Mismatch"
            continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions), key=predictions.count)]

# --- Main Batch Processing Pipeline ---
def main(patient_list, audio_dir):
    """Orchestrates the loading of models and the analysis for a list of patients."""
    # --- PRE-LOADING ALL MODELS (THE OPTIMIZATION) ---
    print("--- Pre-loading all models and resources. Please wait... ---")
    loaded_resources = {}
    all_configs = [CLASSIFIER_MODEL_CONFIG] + EXPERT_PIPELINE_CONFIG
    custom_objects = {"GroupNormalization": GroupNormalization}
    for config in all_configs:
        try:
            model = load_model(config['model_path'], custom_objects=custom_objects if config.get('has_custom_layer') else None, compile=False)
            mean, std = np.load(config['mean_path']), np.load(config['std_path'])
            loaded_resources[config['name']] = {**config, 'model': model, 'mean': mean, 'std': std}
        except Exception as e:
            print(f"❌ Critical Error: Failed to load resources for model '{config['name']}'.\nError: {e}\nExiting.")
            return
    print("✅ All models loaded successfully.")
    
    # --- BATCH PROCESSING LOOP ---
    print(f"\n--- 🩺 Starting Batch Diagnostic Run for {len(patient_list)} Patients ---")
    final_results = []
    for patient_id in patient_list:
        print(f"\n--- Analyzing Patient ID: {patient_id} ---")
        patient_report = {"Patient ID": patient_id}
        patient_files = get_patient_files(patient_id, audio_dir)

        # 1. Run the primary classifier
        primary_resources = loaded_resources[CLASSIFIER_MODEL_CONFIG['name']]
        patient_report["Classified Stage"] = predict_with_primary_classifier(patient_files, primary_resources)
        
        # 2. Run the expert model pipeline
        for expert_config in EXPERT_PIPELINE_CONFIG:
            expert_resources = loaded_resources[expert_config['name']]
            column_name = expert_config['name'].replace("Early Stage", "Stage").replace("Healthy", "H")
            patient_report[column_name] = predict_with_expert_model(patient_files, expert_resources)
            
        final_results.append(patient_report)

    # --- FINAL SUMMARY TABLE ---
    print("\n\n" + "="*90 + "\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n" + "="*90)
    if not final_results:
        print("No results were generated.")
    else:
        results_df = pd.DataFrame(final_results)
        print(results_df.to_string(index=False))
    print("="*90)


if __name__ == "__main__":
    main(PATIENTS_TO_ANALYZE, NEW_AUDIO_DIR)

--- Pre-loading all models and resources. Please wait... ---
✅ All models loaded successfully.

--- 🩺 Starting Batch Diagnostic Run for 64 Patients ---

--- Analyzing Patient ID: 104 ---

--- Analyzing Patient ID: 106 ---

--- Analyzing Patient ID: 107 ---

--- Analyzing Patient ID: 109 ---

--- Analyzing Patient ID: 110 ---

--- Analyzing Patient ID: 112 ---

--- Analyzing Patient ID: 113 ---

--- Analyzing Patient ID: 114 ---

--- Analyzing Patient ID: 117 ---

--- Analyzing Patient ID: 118 ---

--- Analyzing Patient ID: 120 ---

--- Analyzing Patient ID: 124 ---

--- Analyzing Patient ID: 128 ---

--- Analyzing Patient ID: 130 ---

--- Analyzing Patient ID: 132 ---

--- Analyzing Patient ID: 133 ---

--- Analyzing Patient ID: 134 ---

--- Analyzing Patient ID: 138 ---

--- Analyzing Patient ID: 139 ---

--- Analyzing Patient ID: 141 ---

--- Analyzing Patient ID: 142 ---

--- Analyzing Patient ID: 145 ---

--- Analyzing Patient ID: 146 ---

--- Analyzing Patient ID: 147 ---

--- Ana

Predecting with respect to classification

In [5]:
# =========================================================================
#
#       COPD Diagnostic Funnel (Corrected and Final Version)
#
# This script first classifies the patient's stage, then runs ONE
# targeted expert model to check for signs of progression. This version
# contains fixes for all previous syntax and logic errors.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
tf.get_logger().setLevel('ERROR')

# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENTS_TO_ANALYZE = ["104", "106", "107", "109", "223"]

# Configuration for the primary classifier
CLASSIFIER_CONFIG = {
    "name": "Primary Classifier",
    "model_path": os.path.join(MODELS_BASE_PATH, "best_copd_model_finetuned.h5"),
    "mean_path": os.path.join(MODELS_BASE_PATH, "mfcc_training_mean.npy"),
    "std_path": os.path.join(MODELS_BASE_PATH, "mfcc_training_std.npy"),
    "labels": ['COPD0','COPD1','COPD2','COPD3','COPD4'],
    "n_features": 20, "max_len": 150, "feature_type": "mfcc"
}

# Configuration for the expert models, indexed by their BASE stage
EXPERT_CONFIGS = {
    'COPD0': {"name": "Expert 0vs1", "model_path": os.path.join(MODELS_BASE_PATH,"testing_0_1.keras"), "mean_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_mean.npy"), "std_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_0_1_mel20_std.npy"), "labels": {0:'COPD0', 1:'COPD1'}},
    'COPD1': {"name": "Expert 1vs2", "model_path": os.path.join(MODELS_BASE_PATH,"testing_1_2.keras"), "mean_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_mean.npy"), "std_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_1_2_mel20_std.npy"), "labels": {0:'COPD1', 1:'COPD2'}},
    'COPD2': {"name": "Expert 2vs3", "model_path": os.path.join(MODELS_BASE_PATH,"testing_2_3.keras"), "mean_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_mean.npy"), "std_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_2_3_mel20_std.npy"), "labels": {0:'COPD2', 1:'COPD3'}},
    'COPD3': {"name": "Expert 3vs4", "model_path": os.path.join(MODELS_BASE_PATH,"testing_3_4.keras"), "mean_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_mean.npy"), "std_path": os.path.join(MODELS_BASE_PATH,"logmel_testing_3_4_mel20_std.npy"), "labels": {0:'COPD3', 1:'COPD4'}}
}
# Universal expert model parameters
for key in EXPERT_CONFIGS: EXPERT_CONFIGS[key].update({"n_mels": 20, "max_len": 150, "feature_type": "logmel", "has_custom_layer": True})


# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs)
        self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1]; self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'); self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros')
    def call(self, inputs):
        s=tf.shape(inputs); N,H,W,C=s[0],s[1],s[2],s[3]; gs=C//self.groups; r=tf.reshape(inputs,[N,H,W,self.groups,gs]); m,v=tf.nn.moments(r,[1,2,4],keepdims=True); n=(r-m)/tf.sqrt(v+self.epsilon); return tf.reshape(n,s)*self.gamma+self.beta

def get_prediction(patient_files, resources):
    """Unified prediction function that handles both model types."""
    if not patient_files or resources is None: return "Error"
    predictions = []
    is_multiclass = isinstance(resources['labels'], list)
    
    for fpath in patient_files:
        try:
            if resources['feature_type'] == 'mfcc':
                y, sr = librosa.load(fpath, sr=16000)
                feature = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=resources['n_features'])
            else: # logmel
                y, sr = librosa.load(fpath, sr=None)
                feature = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=resources['n_mels']))

            if feature.shape[1] < resources['max_len']: feature = np.pad(feature, ((0,0), (0, resources['max_len'] - feature.shape[1])))
            else: feature = feature[:, :resources['max_len']]

            norm_feature = (feature - resources['mean']) / (resources['std'] + 1e-6)
            final_input = norm_feature[np.newaxis, ..., np.newaxis]
            probs = resources['model'].predict(final_input, verbose=0)[0]

            if is_multiclass: predictions.append(np.argmax(probs))
            else: predictions.append(1 if probs[0] >= 0.5 else 0)
        except Exception as e:
            # print(f"Warning on file {os.path.basename(fpath)}: {e}")
            continue
            
    if not predictions: return "Processing Error"
    
    majority_vote_idx = max(set(predictions), key=predictions.count)
    return resources['labels'][majority_vote_idx]


# --- Main Diagnostic Funnel ---
def main(patient_list, audio_dir):
    """Orchestrates loading models and the targeted analysis workflow."""
    print("--- Pre-loading all models and resources... ---")
    
    # Pre-load all resources into a single dictionary
    all_resources = {}
    all_configs = [CLASSIFIER_CONFIG] + list(EXPERT_CONFIGS.values())
    custom_objects = {"GroupNormalization": GroupNormalization}

    for config in all_configs:
        try:
            model = load_model(os.path.join(MODELS_BASE_PATH, config['model_path']), custom_objects=custom_objects if config.get('has_custom_layer', False) else None, compile=False)
            mean = np.load(os.path.join(MODELS_BASE_PATH, config['mean_path']))
            std = np.load(os.path.join(MODELS_BASE_PATH, config['std_path']))
            all_resources[config['name']] = {**config, 'model': model, 'mean': mean, 'std': std}
            print(f"  ✅ Loaded: {config['name']}")
        except Exception as e:
            print(f"  ❌ FAILED to load '{config['name']}': {e}"); all_resources[config['name']] = None

    print("--- Model loading complete. ---\n")
    
    final_results = []
    for patient_id in patient_list:
        print("="*60 + f"\n--- Analyzing Patient ID: {patient_id} ---")
        patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]
        
        # 1. Get Primary Classification
        clf_resources = all_resources.get(CLASSIFIER_CONFIG['name'])
        classified_stage = get_prediction(patient_files, clf_resources)
        print(f"  -> Primary Classifier Verdict: {classified_stage}")
        
        # 2. Conditional Expert Analysis
        expert_verdict = "N/A" # Default verdict
        if classified_stage in EXPERT_CONFIGS:
            relevant_expert_key = classified_stage
            expert_resources = all_resources.get(EXPERT_CONFIGS[relevant_expert_key]['name'])
            if expert_resources:
                print(f"  -> Consulting '{expert_resources['name']}' to assess progression...")
                expert_verdict = get_prediction(patient_files, expert_resources)
            else:
                expert_verdict = "Expert Model Load Failed"
        elif classified_stage == "COPD4":
             expert_verdict = "Final Stage (No Progression Check)"

        print(f"  -> Expert Analysis Result: {expert_verdict}")
        
        final_results.append({
            "Patient ID": patient_id,
            "Classified Stage": classified_stage,
            "Progression Check": expert_verdict
        })

    # Display final summary table
    print("\n\n" + "="*90 + "\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n" + "="*90)
    print(pd.DataFrame(final_results).to_string(index=False))
    print("="*90)

if __name__ == "__main__":
    main(sorted(list(set(PATIENTS_TO_ANALYZE))), NEW_AUDIO_DIR)

--- Pre-loading all models and resources... ---
  ✅ Loaded: Primary Classifier
  ✅ Loaded: Expert 0vs1
  ✅ Loaded: Expert 1vs2
  ✅ Loaded: Expert 2vs3
  ✅ Loaded: Expert 3vs4
--- Model loading complete. ---

--- Analyzing Patient ID: 104 ---
  -> Primary Classifier Verdict: COPD4
  -> Expert Analysis Result: Final Stage (No Progression Check)
--- Analyzing Patient ID: 106 ---
  -> Primary Classifier Verdict: COPD3
  -> Consulting 'Expert 3vs4' to assess progression...
  -> Expert Analysis Result: COPD3
--- Analyzing Patient ID: 107 ---
  -> Primary Classifier Verdict: COPD3
  -> Consulting 'Expert 3vs4' to assess progression...
  -> Expert Analysis Result: COPD3
--- Analyzing Patient ID: 109 ---
  -> Primary Classifier Verdict: COPD4
  -> Expert Analysis Result: Final Stage (No Progression Check)
--- Analyzing Patient ID: 223 ---
  -> Primary Classifier Verdict: COPD4
  -> Expert Analysis Result: Final Stage (No Progression Check)


--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---
Patient ID C

In percentage form

In [2]:
# =========================================================================
#
#       Intelligent Diagnostic Pipeline with Percentage Progression Risk
#
# This script first classifies the patient's stage, then consults ONE
# targeted expert model to calculate and display the percentage risk
# of progression to the next stage.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
tf.get_logger().setLevel('ERROR')

# --- Step 1: Central Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENTS_TO_ANALYZE = ["104", "106", "107", "109", "110", "112", "113", "114", "117", "118", "120", "124", "128", "130", "132", "133", "134", "138", "139", "141", "142", "145", "146", "147", "151", "154", "155", "156", "157", "158", "160", "162", "163", "166", "170", "172", "174", "175", "176", "177", "178", "180", "181", "185", "186", "189", "192", "193", "195", "198", "199", "200", "203", "204", "205", "207", "211", "212", "213", "218", "220", "221", "222", "223"]

# Configuration for the primary classifier
CLASSIFIER_CONFIG = { "name":"Primary Classifier", "model_path":os.path.join(MODELS_BASE_PATH,"best_copd_model_finetuned.h5"), "mean_path":os.path.join(MODELS_BASE_PATH,"mfcc_training_mean.npy"), "std_path":os.path.join(MODELS_BASE_PATH,"mfcc_training_std.npy"), "labels":['COPD0','COPD1','COPD2','COPD3','COPD4'],"feature_type":"mfcc", "n_features": 20, "max_len": 150}

# Configuration for the expert models, indexed by their BASE stage
EXPERT_CONFIGS = {
    'COPD0': {"name":"Expert 0vs1", "model_path":"testing_0_1.keras", "mean_path":"logmel_testing_0_1_mel20_mean.npy", "std_path":"logmel_testing_0_1_mel20_std.npy", "labels":{0:'COPD0', 1:'COPD1'},"n_mels":20},
    'COPD1': {"name":"Expert 1vs2", "model_path":"testing_1_2.keras", "mean_path":"logmel_testing_1_2_mel20_mean.npy", "std_path":"logmel_testing_1_2_mel20_std.npy", "labels":{0:'COPD1', 1:'COPD2'},"n_mels":20},
    'COPD2': {"name":"Expert 2vs3", "model_path":"testing_2_3.keras", "mean_path":"logmel_testing_2_3_mel20_mean.npy", "std_path":"logmel_testing_2_3_mel20_std.npy", "labels":{0:'COPD2', 1:'COPD3'},"n_mels":20},
    'COPD3': {"name":"Expert 3vs4", "model_path":"testing_3_4.keras", "mean_path":"logmel_testing_3_4_mel20_mean.npy", "std_path":"logmel_testing_3_4_mel20_std.npy", "labels":{0:'COPD3', 1:'COPD4'},"n_mels":20}
}
for key in EXPERT_CONFIGS: EXPERT_CONFIGS[key].update({"max_len": 150, "feature_type": "logmel", "has_custom_layer": True})


# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self,g=4,e=1e-5,**kwargs): super().__init__(**kwargs);self.g,self.e=g,e
    def build(self,s): d=s[-1];self.gm,self.bt=self.add_weight(n='g',s=(1,1,1,d),i='o'),self.add_weight(n='b',s=(1,1,1,d),i='z')
    def call(self,i):s=tf.shape(i);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.g;r=tf.reshape(i,[N,H,W,self.g,gs]);m,v=tf.nn.moments(r,[1,2,4],k=1);n=(r-m)/tf.sqrt(v+self.e);return tf.reshape(n,s)*self.gm+self.bt

def get_classifier_prediction(patient_files, resources):
    """Predicts a stage from the primary multi-class classifier."""
    if not patient_files or resources is None: return "Error"
    predictions = []
    for fpath in patient_files:
        try:
            y,sr=librosa.load(fpath,sr=16000); feature=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=resources['n_features'])
            if feature.shape[1]<resources['max_len']:feature=np.pad(feature,((0,0),(0,resources['max_len']-feature.shape[1])))
            else:feature=feature[:,:resources['max_len']]
            norm=(feature-resources['mean'])/(resources['std']+1e-6);final=norm[np.newaxis,...,np.newaxis]
            probs=resources['model'].predict(final,verbose=0)[0]
            predictions.append(np.argmax(probs))
        except: continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions),key=predictions.count)]

# --- MODIFIED Function for Progression Risk ---
def get_progression_risk_percentage(patient_files, resources):
    """Calculates and returns the average progression risk as a formatted string."""
    if not patient_files or resources is None: return "Error"
    progression_scores = []
    for fpath in patient_files:
        try:
            y,sr=librosa.load(fpath,sr=None); feature=librosa.power_to_db(librosa.feature.melspectrogram(y=y,sr=sr,n_mels=resources['n_mels']))
            if feature.shape[1]<resources['max_len']:feature=np.pad(feature,((0,0),(0,resources['max_len']-feature.shape[1])))
            else:feature=feature[:,:resources['max_len']]
            norm=(feature-resources['mean'])/(resources['std']+1e-6);final=norm[np.newaxis,...,np.newaxis]
            raw_prob=resources['model'].predict(final,verbose=0)[0][0]
            progression_scores.append(raw_prob)
        except: continue
    if not progression_scores: return "Processing Error"
    return f"{np.mean(progression_scores)*100:.2f}%"

# --- Main Diagnostic Funnel ---
def main(patient_list, audio_dir):
    """Orchestrates loading models and the targeted analysis workflow."""
    print("--- Pre-loading all models and resources... ---")
    all_resources = {}
    all_configs = [CLASSIFIER_CONFIG] + list(EXPERT_CONFIGS.values())
    custom_objects = {"GroupNormalization": GroupNormalization}
    for config in all_configs:
        try:
            model = load_model(os.path.join(MODELS_BASE_PATH, config['model_path']), custom_objects=custom_objects if config.get('has_custom_layer', False) else None, compile=False)
            all_resources[config['name']] = {"model":model, "mean":np.load(os.path.join(MODELS_BASE_PATH,config['mean_path'])), "std":np.load(os.path.join(MODELS_BASE_PATH,config['std_path'])), **config}
            print(f"  ✅ Loaded: {config['name']}")
        except Exception as e: print(f"  ❌ FAILED to load '{config['name']}': {e}"); all_resources[config['name']] = None

    print("--- Model loading complete. ---\n")
    final_results = []
    for patient_id in patient_list:
        print("="*60 + f"\n--- Analyzing Patient ID: {patient_id} ---")
        patient_files = [os.path.join(audio_dir,f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]
        
        clf_resources = all_resources.get(CLASSIFIER_CONFIG['name'])
        classified_stage = get_classifier_prediction(patient_files, clf_resources)
        print(f"  -> Primary Classifier Verdict: {classified_stage}")
        
        progression_risk = "N/A"
        progression_check_label = "Progression Check"
        if classified_stage in EXPERT_CONFIGS:
            expert_config = EXPERT_CONFIGS[classified_stage]
            expert_resources = all_resources.get(expert_config['name'])
            if expert_resources:
                next_stage = expert_config['labels'][1]
                progression_check_label = f"Risk of -> {next_stage}"
                progression_risk = get_progression_risk_percentage(patient_files, expert_resources)
            else:
                progression_risk = "Expert Model Load Failed"
        elif classified_stage == "COPD4":
             progression_risk = "Final Stage"

        print(f"  -> Progression Risk Assessment: {progression_risk}")
        final_results.append({"Patient ID": patient_id, "Classified Stage": classified_stage, "Progression Analysis": progression_risk})

    print("\n\n" + "="*90 + "\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n" + "="*90)
    print(pd.DataFrame(final_results).to_string(index=False))
    print("="*90)

if __name__ == "__main__":
    main(sorted(list(set(PATIENTS_TO_ANALYZE))), NEW_AUDIO_DIR)

--- Pre-loading all models and resources... ---
  ✅ Loaded: Primary Classifier
  ✅ Loaded: Expert 0vs1
  ✅ Loaded: Expert 1vs2
  ✅ Loaded: Expert 2vs3
  ❌ FAILED to load 'Expert 3vs4': <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential_3', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 127832628814016}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 20, 150, 1], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_3'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Conv2D', 'config': {'name': 

something

In [5]:
# =========================================================================
#
#       Universal Expert Model Trainer (DEFINITIVE, SYNTAX-CORRECTED)
#
# =========================================================================
import os, sys, numpy as np, pandas as pd, librosa, librosa.effects, tensorflow as tf, random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ===============================================================
#                !!! TASK CONFIGURATION !!!
# ===============================================================
CLASSES_TO_TRAIN = ['COPD0', 'COPD1']
MODEL_TO_SAVE_AS = "testing_0_1.keras"
# ===============================================================

# --- General Configuration ---
LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"
N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
INITIAL_LEARNING_RATE = 0.0001

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1]; self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'); self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros'); super(GroupNormalization, self).build(input_shape)
    def call(self, inputs):
        s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta
    def get_config(self):
        config=super(GroupNormalization, self).get_config();config.update({"groups":self.groups,"epsilon":self.epsilon});return config

def extract_log_mel_spectrogram(path, do_augment=False):
    y,sr=librosa.load(path,sr=None)
    if do_augment:
        if random.random()<0.5:y=librosa.effects.time_stretch(y,rate=random.uniform(0.9,1.1))
        else: y=librosa.effects.pitch_shift(y,sr=sr,n_steps=random.randint(-2,2))
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=N_MELS);log_mel=librosa.power_to_db(mel)
    if log_mel.shape[1]<MAX_LEN: log_mel=np.pad(log_mel,((0,0),(0,MAX_LEN-log_mel.shape[1])))
    else: log_mel=log_mel[:,:MAX_LEN]
    return log_mel

# --- Data Preparation ---
TASK_NAME=f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
print(f"\n{'='*20} TRAINING: {TASK_NAME} {'='*20}")
df=pd.read_excel(LABEL_PATH);df_task=df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
df_task['label']=df_task['Diagnosis'].apply(lambda x:0 if x==CLASSES_TO_TRAIN[0] else 1);label_dict=dict(zip(df_task["Patient ID"],df_task["label"]))
df0,df1=df_task[df_task.label==0],df_task[df_task.label==1];min_size=min(len(df0),len(df1))
df_balanced=pd.concat([df0.sample(n=min_size,random_state=42),df1.sample(n=min_size,random_state=42)])
p_ids,p_labels=list(df_balanced["Patient ID"]),list(df_balanced["label"])
train_pids,test_pids,_,_=train_test_split(p_ids,p_labels,test_size=0.25,random_state=42,stratify=p_labels)
X_train,y_train,X_test,y_test=[],[],[],[]
for pid in p_ids:
    paths=[os.path.join(AUDIO_DIR,f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        try:
            if pid in train_pids:
                X_train.append(extract_log_mel_spectrogram(path));y_train.append(label_dict[pid])
                for _ in range(3):X_train.append(extract_log_mel_spectrogram(path,do_augment=True));y_train.append(label_dict[pid])
            elif pid in test_pids:X_test.append(extract_log_mel_spectrogram(path));y_test.append(label_dict[pid])
        except Exception as e:print(f"Warn: {path}: {e}")
X_train_raw=np.array(X_train);mean_val,std_val=np.mean(X_train_raw,axis=0),np.std(X_train_raw,axis=0)
model_name=MODEL_TO_SAVE_AS.replace('.keras','');np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_mean.npy"),mean_val);np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_std.npy"),std_val)
X_train=(X_train_raw-mean_val)/(std_val+1e-6);X_test=(np.array(X_test)-mean_val)/(std_val+1e-6)
X_train,X_test=X_train[...,np.newaxis],X_test[...,np.newaxis];y_train,y_test=np.array(y_train),np.array(y_test)

# --- CORRECTED Model Architecture ---
model = Sequential([
    Input(shape=(N_MELS, MAX_LEN, 1)),
    
    Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(), # Each layer is now correctly separated by a comma
    Dropout(0.3),
    
    Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    Flatten(),
    Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# --- Model Training ---
print("\nStarting training...")
model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
cbs = [EarlyStopping('val_loss',patience=10,restore_best_weights=True), ModelCheckpoint(model_path,save_best_only=True, monitor='val_accuracy'), ReduceLROnPlateau(patience=4)]
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)

# --- Final Model Evaluation ---
model.load_weights(model_path)
_, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Final Test Accuracy for {TASK_NAME}: {accuracy * 100:.2f}% ---")
print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== TRAINING: COPD0_vs_COPD1 ====================


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_2           │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_3           │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.5750 - loss: 1.4924 - val_accuracy: 0.6111 - val_loss: 1.0229 - learning_rate: 1.0000e-04
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.5912 - loss: 1.1924 - val_accuracy: 0.5833 - val_loss: 0.9476 - learning_rate: 1.0000e-04
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.6861 - loss: 0.9816 - val_accuracy: 0.6111 - val_loss: 0.9385 - learning_rate: 1.0000e-04
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.7198 - loss: 0.8187 - val_accuracy: 0.5833 - val_loss: 0.8882 - learning_rate: 1.0000e-04
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.7017 - loss: 0.7591 - val_accuracy: 0.6389 - val_loss: 0.9695 - learning_rate: 1.0000e-04
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.7921 - loss: 0.6484 - val_accuracy: 0.6667 - val_loss: 0.9214 - learning_rate: 1.0000e-04
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/st

In [9]:
# =========================================================================
#
#       Universal Expert Model Trainer (DEFINITIVE, SYNTAX-CORRECTED)
#
# =========================================================================
import os, sys, numpy as np, pandas as pd, librosa, librosa.effects, tensorflow as tf, random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ===============================================================
#                !!! TASK CONFIGURATION !!!
# ===============================================================
CLASSES_TO_TRAIN = ['COPD1', 'COPD2']
MODEL_TO_SAVE_AS = "testing_1_2.keras"
# ===============================================================

# --- General Configuration ---
LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"
N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
INITIAL_LEARNING_RATE = 0.0001

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1]; self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'); self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros'); super(GroupNormalization, self).build(input_shape)
    def call(self, inputs):
        s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta
    def get_config(self):
        config=super(GroupNormalization, self).get_config();config.update({"groups":self.groups,"epsilon":self.epsilon});return config

def extract_log_mel_spectrogram(path, do_augment=False):
    y,sr=librosa.load(path,sr=None)
    if do_augment:
        if random.random()<0.5:y=librosa.effects.time_stretch(y,rate=random.uniform(0.9,1.1))
        else: y=librosa.effects.pitch_shift(y,sr=sr,n_steps=random.randint(-2,2))
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=N_MELS);log_mel=librosa.power_to_db(mel)
    if log_mel.shape[1]<MAX_LEN: log_mel=np.pad(log_mel,((0,0),(0,MAX_LEN-log_mel.shape[1])))
    else: log_mel=log_mel[:,:MAX_LEN]
    return log_mel

# --- Data Preparation ---
TASK_NAME=f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
print(f"\n{'='*20} TRAINING: {TASK_NAME} {'='*20}")
df=pd.read_excel(LABEL_PATH);df_task=df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
df_task['label']=df_task['Diagnosis'].apply(lambda x:0 if x==CLASSES_TO_TRAIN[0] else 1);label_dict=dict(zip(df_task["Patient ID"],df_task["label"]))
df0,df1=df_task[df_task.label==0],df_task[df_task.label==1];min_size=min(len(df0),len(df1))
df_balanced=pd.concat([df0.sample(n=min_size,random_state=42),df1.sample(n=min_size,random_state=42)])
p_ids,p_labels=list(df_balanced["Patient ID"]),list(df_balanced["label"])
train_pids,test_pids,_,_=train_test_split(p_ids,p_labels,test_size=0.25,random_state=42,stratify=p_labels)
X_train,y_train,X_test,y_test=[],[],[],[]
for pid in p_ids:
    paths=[os.path.join(AUDIO_DIR,f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        try:
            if pid in train_pids:
                X_train.append(extract_log_mel_spectrogram(path));y_train.append(label_dict[pid])
                for _ in range(3):X_train.append(extract_log_mel_spectrogram(path,do_augment=True));y_train.append(label_dict[pid])
            elif pid in test_pids:X_test.append(extract_log_mel_spectrogram(path));y_test.append(label_dict[pid])
        except Exception as e:print(f"Warn: {path}: {e}")
X_train_raw=np.array(X_train);mean_val,std_val=np.mean(X_train_raw,axis=0),np.std(X_train_raw,axis=0)
model_name=MODEL_TO_SAVE_AS.replace('.keras','');np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_mean.npy"),mean_val);np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_std.npy"),std_val)
X_train=(X_train_raw-mean_val)/(std_val+1e-6);X_test=(np.array(X_test)-mean_val)/(std_val+1e-6)
X_train,X_test=X_train[...,np.newaxis],X_test[...,np.newaxis];y_train,y_test=np.array(y_train),np.array(y_test)

# --- CORRECTED Model Architecture ---
model = Sequential([
    Input(shape=(N_MELS, MAX_LEN, 1)),
    
    Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(), # Each layer is now correctly separated by a comma
    Dropout(0.3),
    
    Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    Flatten(),
    Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# --- Model Training ---
print("\nStarting training...")
model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
cbs = [EarlyStopping('val_loss',patience=10,restore_best_weights=True), ModelCheckpoint(model_path,save_best_only=True, monitor='val_accuracy'), ReduceLROnPlateau(patience=4)]
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)

# --- Final Model Evaluation ---
model.load_weights(model_path)
_, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Final Test Accuracy for {TASK_NAME}: {accuracy * 100:.2f}% ---")
print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== TRAINING: COPD1_vs_COPD2 ====================


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_10          │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_11          │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.4951 - loss: 1.7303 - val_accuracy: 0.3889 - val_loss: 1.4878 - learning_rate: 1.0000e-04
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.5608 - loss: 1.2429 - val_accuracy: 0.3889 - val_loss: 1.5117 - learning_rate: 1.0000e-04
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.6943 - loss: 0.8131 - val_accuracy: 0.3889 - val_loss: 1.5540 - learning_rate: 1.0000e-04
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.7992 - loss: 0.5951 - val_accuracy: 0.3889 - val_loss: 1.6020 - learning_rate: 1.0000e-04
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.8026 - loss: 0.5790 - val_accuracy: 0.4167 - val_loss: 1.5710 - learning_rate: 1.0000e-04
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.8274 - loss: 0.5683 - val_accuracy: 0.4167 - val_loss: 1.5678 - learning_rate: 1.0000e-05
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/st

In [7]:
# =========================================================================
#
#       Universal Expert Model Trainer (DEFINITIVE, SYNTAX-CORRECTED)
#
# =========================================================================
import os, sys, numpy as np, pandas as pd, librosa, librosa.effects, tensorflow as tf, random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ===============================================================
#                !!! TASK CONFIGURATION !!!
# ===============================================================
CLASSES_TO_TRAIN = ['COPD2', 'COPD3']
MODEL_TO_SAVE_AS = "testing_2_3.keras"
# ===============================================================

# --- General Configuration ---
LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"
N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
INITIAL_LEARNING_RATE = 0.0001

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1]; self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'); self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros'); super(GroupNormalization, self).build(input_shape)
    def call(self, inputs):
        s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta
    def get_config(self):
        config=super(GroupNormalization, self).get_config();config.update({"groups":self.groups,"epsilon":self.epsilon});return config

def extract_log_mel_spectrogram(path, do_augment=False):
    y,sr=librosa.load(path,sr=None)
    if do_augment:
        if random.random()<0.5:y=librosa.effects.time_stretch(y,rate=random.uniform(0.9,1.1))
        else: y=librosa.effects.pitch_shift(y,sr=sr,n_steps=random.randint(-2,2))
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=N_MELS);log_mel=librosa.power_to_db(mel)
    if log_mel.shape[1]<MAX_LEN: log_mel=np.pad(log_mel,((0,0),(0,MAX_LEN-log_mel.shape[1])))
    else: log_mel=log_mel[:,:MAX_LEN]
    return log_mel

# --- Data Preparation ---
TASK_NAME=f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
print(f"\n{'='*20} TRAINING: {TASK_NAME} {'='*20}")
df=pd.read_excel(LABEL_PATH);df_task=df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
df_task['label']=df_task['Diagnosis'].apply(lambda x:0 if x==CLASSES_TO_TRAIN[0] else 1);label_dict=dict(zip(df_task["Patient ID"],df_task["label"]))
df0,df1=df_task[df_task.label==0],df_task[df_task.label==1];min_size=min(len(df0),len(df1))
df_balanced=pd.concat([df0.sample(n=min_size,random_state=42),df1.sample(n=min_size,random_state=42)])
p_ids,p_labels=list(df_balanced["Patient ID"]),list(df_balanced["label"])
train_pids,test_pids,_,_=train_test_split(p_ids,p_labels,test_size=0.25,random_state=42,stratify=p_labels)
X_train,y_train,X_test,y_test=[],[],[],[]
for pid in p_ids:
    paths=[os.path.join(AUDIO_DIR,f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        try:
            if pid in train_pids:
                X_train.append(extract_log_mel_spectrogram(path));y_train.append(label_dict[pid])
                for _ in range(3):X_train.append(extract_log_mel_spectrogram(path,do_augment=True));y_train.append(label_dict[pid])
            elif pid in test_pids:X_test.append(extract_log_mel_spectrogram(path));y_test.append(label_dict[pid])
        except Exception as e:print(f"Warn: {path}: {e}")
X_train_raw=np.array(X_train);mean_val,std_val=np.mean(X_train_raw,axis=0),np.std(X_train_raw,axis=0)
model_name=MODEL_TO_SAVE_AS.replace('.keras','');np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_mean.npy"),mean_val);np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_std.npy"),std_val)
X_train=(X_train_raw-mean_val)/(std_val+1e-6);X_test=(np.array(X_test)-mean_val)/(std_val+1e-6)
X_train,X_test=X_train[...,np.newaxis],X_test[...,np.newaxis];y_train,y_test=np.array(y_train),np.array(y_test)

# --- CORRECTED Model Architecture ---
model = Sequential([
    Input(shape=(N_MELS, MAX_LEN, 1)),
    
    Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(), # Each layer is now correctly separated by a comma
    Dropout(0.3),
    
    Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    Flatten(),
    Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# --- Model Training ---
print("\nStarting training...")
model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
cbs = [EarlyStopping('val_loss',patience=10,restore_best_weights=True), ModelCheckpoint(model_path,save_best_only=True, monitor='val_accuracy'), ReduceLROnPlateau(patience=4)]
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)

# --- Final Model Evaluation ---
model.load_weights(model_path)
_, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Final Test Accuracy for {TASK_NAME}: {accuracy * 100:.2f}% ---")
print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== TRAINING: COPD2_vs_COPD3 ====================


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_6           │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_7           │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.5026 - loss: 1.7218 - val_accuracy: 0.5417 - val_loss: 1.4139 - learning_rate: 1.0000e-04
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.7099 - loss: 0.9186 - val_accuracy: 0.5625 - val_loss: 0.9377 - learning_rate: 1.0000e-04
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.7608 - loss: 0.7242 - val_accuracy: 0.6250 - val_loss: 1.0857 - learning_rate: 1.0000e-04
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.7496 - loss: 0.7147 - val_accuracy: 0.5625 - val_loss: 1.0073 - learning_rate: 1.0000e-04
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.8027 - loss: 0.6246 - val_accuracy: 0.5625 - val_loss: 1.0609 - learning_rate: 1.0000e-04
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.8021 - loss: 0.6012 - val_accuracy: 0.6042 - val_loss: 1.0400 - learning_rate: 1.0000e-04
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/st

In [8]:
# =========================================================================
#
#       Universal Expert Model Trainer (DEFINITIVE, SYNTAX-CORRECTED)
#
# =========================================================================
import os, sys, numpy as np, pandas as pd, librosa, librosa.effects, tensorflow as tf, random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ===============================================================
#                !!! TASK CONFIGURATION !!!
# ===============================================================
CLASSES_TO_TRAIN = ['COPD3', 'COPD4']
MODEL_TO_SAVE_AS = "testing_3_4.keras"
# ===============================================================

# --- General Configuration ---
LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"
N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
INITIAL_LEARNING_RATE = 0.0001

# --- Helper Layers & Functions ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon = groups, epsilon
    def build(self, input_shape):
        dim=input_shape[-1]; self.gamma=self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'); self.beta=self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros'); super(GroupNormalization, self).build(input_shape)
    def call(self, inputs):
        s=tf.shape(inputs);N,H,W,C=s[0],s[1],s[2],s[3];gs=C//self.groups;r=tf.reshape(inputs,[N,H,W,self.groups,gs]);m,v=tf.nn.moments(r,[1,2,4],keepdims=True);n=(r-m)/tf.sqrt(v+self.epsilon);return tf.reshape(n,s)*self.gamma+self.beta
    def get_config(self):
        config=super(GroupNormalization, self).get_config();config.update({"groups":self.groups,"epsilon":self.epsilon});return config

def extract_log_mel_spectrogram(path, do_augment=False):
    y,sr=librosa.load(path,sr=None)
    if do_augment:
        if random.random()<0.5:y=librosa.effects.time_stretch(y,rate=random.uniform(0.9,1.1))
        else: y=librosa.effects.pitch_shift(y,sr=sr,n_steps=random.randint(-2,2))
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=N_MELS);log_mel=librosa.power_to_db(mel)
    if log_mel.shape[1]<MAX_LEN: log_mel=np.pad(log_mel,((0,0),(0,MAX_LEN-log_mel.shape[1])))
    else: log_mel=log_mel[:,:MAX_LEN]
    return log_mel

# --- Data Preparation ---
TASK_NAME=f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
print(f"\n{'='*20} TRAINING: {TASK_NAME} {'='*20}")
df=pd.read_excel(LABEL_PATH);df_task=df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
df_task['label']=df_task['Diagnosis'].apply(lambda x:0 if x==CLASSES_TO_TRAIN[0] else 1);label_dict=dict(zip(df_task["Patient ID"],df_task["label"]))
df0,df1=df_task[df_task.label==0],df_task[df_task.label==1];min_size=min(len(df0),len(df1))
df_balanced=pd.concat([df0.sample(n=min_size,random_state=42),df1.sample(n=min_size,random_state=42)])
p_ids,p_labels=list(df_balanced["Patient ID"]),list(df_balanced["label"])
train_pids,test_pids,_,_=train_test_split(p_ids,p_labels,test_size=0.25,random_state=42,stratify=p_labels)
X_train,y_train,X_test,y_test=[],[],[],[]
for pid in p_ids:
    paths=[os.path.join(AUDIO_DIR,f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        try:
            if pid in train_pids:
                X_train.append(extract_log_mel_spectrogram(path));y_train.append(label_dict[pid])
                for _ in range(3):X_train.append(extract_log_mel_spectrogram(path,do_augment=True));y_train.append(label_dict[pid])
            elif pid in test_pids:X_test.append(extract_log_mel_spectrogram(path));y_test.append(label_dict[pid])
        except Exception as e:print(f"Warn: {path}: {e}")
X_train_raw=np.array(X_train);mean_val,std_val=np.mean(X_train_raw,axis=0),np.std(X_train_raw,axis=0)
model_name=MODEL_TO_SAVE_AS.replace('.keras','');np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_mean.npy"),mean_val);np.save(os.path.join(SAVE_DIR,f"logmel_{model_name}_std.npy"),std_val)
X_train=(X_train_raw-mean_val)/(std_val+1e-6);X_test=(np.array(X_test)-mean_val)/(std_val+1e-6)
X_train,X_test=X_train[...,np.newaxis],X_test[...,np.newaxis];y_train,y_test=np.array(y_train),np.array(y_test)

# --- CORRECTED Model Architecture ---
model = Sequential([
    Input(shape=(N_MELS, MAX_LEN, 1)),
    
    Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(), # Each layer is now correctly separated by a comma
    Dropout(0.3),
    
    Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
    GroupNormalization(),
    MaxPooling2D(),
    Dropout(0.3),
    
    Flatten(),
    Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# --- Model Training ---
print("\nStarting training...")
model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
cbs = [EarlyStopping('val_loss',patience=10,restore_best_weights=True), ModelCheckpoint(model_path,save_best_only=True, monitor='val_accuracy'), ReduceLROnPlateau(patience=4)]
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)

# --- Final Model Evaluation ---
model.load_weights(model_path)
_, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n--- Final Test Accuracy for {TASK_NAME}: {accuracy * 100:.2f}% ---")
print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== TRAINING: COPD3_vs_COPD4 ====================


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_8           │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_9           │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 88ms/step - accuracy: 0.5280 - loss: 1.4889 - val_accuracy: 0.4375 - val_loss: 1.2208 - learning_rate: 1.0000e-04
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.5899 - loss: 1.0220 - val_accuracy: 0.4583 - val_loss: 1.0384 - learning_rate: 1.0000e-04
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.6573 - loss: 0.8473 - val_accuracy: 0.5000 - val_loss: 1.0504 - learning_rate: 1.0000e-04
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.7209 - loss: 0.7409 - val_accuracy: 0.4792 - val_loss: 1.0781 - learning_rate: 1.0000e-04
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.7719 - loss: 0.6531 - val_accuracy: 0.5000 - val_loss: 1.1098 - learning_rate: 1.0000e-04
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.7819 - loss: 0.6117 - val_accuracy: 0.5000 - val_loss: 1.1324 - learning_rate: 1.0000e-04
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/st

In [11]:
# =========================================================================
#       COPD Final Diagnostic Pipeline
# =========================================================================
import os, librosa, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import warnings
warnings.filterwarnings('ignore', category=UserWarning); tf.get_logger().setLevel('ERROR')

# --- Configuration ---
NEW_AUDIO_DIR = "./RespiratoryDatabase@TR/"
MODELS_BASE_PATH = "./models/"
PATIENTS_TO_ANALYZE = ["104", "106", "107", "109", "110", "112", "113", "114", "117", "118", "120", "124", "128", "130", "132", "133", "134", "138", "139", "141", "142", "145", "146", "147", "151", "154", "155", "156", "157", "158", "160", "162", "163", "166", "170", "172", "174", "175", "176", "177", "178", "180", "181", "185", "186", "189", "192", "193", "195", "198", "199", "200", "203", "204", "205", "207", "211", "212", "213", "218", "220", "221", "222", "223"]

CLASSIFIER_CONFIG = {"name":"Primary Classifier", "model_path":"best_copd_model_finetuned.h5", "mean_path":"mfcc_training_mean.npy", "std_path":"mfcc_training_std.npy", "labels":['COPD0','COPD1','COPD2','COPD3','COPD4'], "feature_type":"mfcc", "n_features":20, "max_len":150}
EXPERT_CONFIGS = {
    'COPD0': {"name":"Expert 0vs1", "model_path":"testing_0_1.keras", "mean_path":"logmel_testing_0_1_mean.npy", "std_path":"logmel_testing_0_1_std.npy"},
    'COPD1': {"name":"Expert 1vs2", "model_path":"testing_1_2.keras", "mean_path":"logmel_testing_1_2_mel20_mean.npy", "std_path":"logmel_testing_1_2_mel20_std.npy"},
    'COPD2': {"name":"Expert 2vs3", "model_path":"testing_2_3.keras", "mean_path":"logmel_testing_2_3_mel20_mean.npy", "std_path":"logmel_testing_2_3_mel20_std.npy"},
    'COPD3': {"name":"Expert 3vs4", "model_path":"testing_3_4.keras", "mean_path":"logmel_testing_3_4_mel20_mean.npy", "std_path":"logmel_testing_3_4_mel20_std.npy"}
}
for key, val in EXPERT_CONFIGS.items(): val.update({"labels":{0:key, 1:f"COPD{int(key[-1])+1}"}, "n_mels":20, "max_len":150, "feature_type":"logmel", "has_custom_layer":True})

# --- Custom Layer (MUST match the one used in training) ---
class GroupNormalization(Layer):
    def __init__(self, groups=4, epsilon=1e-5, **kwargs):
        super(GroupNormalization, self).__init__(**kwargs); self.groups, self.epsilon=groups,epsilon
    def build(self, input_shape):
        dim = input_shape[-1]; self.gamma, self.beta = self.add_weight(name='gamma',shape=(1,1,1,dim),initializer='ones'), self.add_weight(name='beta',shape=(1,1,1,dim),initializer='zeros'); super(GroupNormalization, self).build(input_shape)
    def call(self, inputs):
        s=tf.shape(inputs); N,H,W,C=s[0],s[1],s[2],s[3]; gs=C//self.groups; r=tf.reshape(inputs,[N,H,W,self.groups,gs]); m,v=tf.nn.moments(r,[1,2,4],keepdims=True); n=(r-m)/tf.sqrt(v+self.epsilon); return tf.reshape(n,s)*self.gamma+self.beta
    def get_config(self):
        config=super(GroupNormalization,self).get_config(); config.update({"groups":self.groups,"epsilon":self.epsilon}); return config

# --- Prediction Logic ---
def get_classifier_prediction(patient_files, resources):
    if not patient_files or resources is None: return "Error"
    predictions=[]
    for fpath in patient_files:
        try:
            y,sr=librosa.load(fpath,sr=16000);feature=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=resources['n_features'])
            if feature.shape[1]<resources['max_len']:feature=np.pad(feature,((0,0),(0,resources['max_len']-feature.shape[1])))
            else:feature=feature[:,:resources['max_len']]
            norm=(feature-resources['mean'])/(resources['std']+1e-6);final=norm[np.newaxis,...,np.newaxis]
            probs=resources['model'].predict(final,verbose=0)[0];predictions.append(np.argmax(probs))
        except: continue
    if not predictions: return "Processing Error"
    return resources['labels'][max(set(predictions),key=predictions.count)]
def get_progression_risk_percentage(patient_files, resources):
    if not patient_files or resources is None: return "Error"
    progression_scores = []
    for fpath in patient_files:
        try:
            y,sr=librosa.load(fpath,sr=None);feature=librosa.power_to_db(librosa.feature.melspectrogram(y=y,sr=sr,n_mels=resources['n_mels']))
            if feature.shape[1]<resources['max_len']:feature=np.pad(feature,((0,0),(0,resources['max_len']-feature.shape[1])))
            else:feature=feature[:,:resources['max_len']]
            norm=(feature-resources['mean'])/(resources['std']+1e-6);final=norm[np.newaxis,...,np.newaxis]
            progression_scores.append(resources['model'].predict(final,verbose=0)[0][0])
        except: continue
    if not progression_scores: return "Processing Error"
    return f"{np.mean(progression_scores)*100:.2f}%"

# --- Main Pipeline ---
def main(patient_list, audio_dir):
    print("--- Pre-loading all models... ---")
    all_resources = {}
    for cfg in [CLASSIFIER_CONFIG] + list(EXPERT_CONFIGS.values()):
        try:
            all_resources[cfg['name']] = {
                "model": load_model(os.path.join(MODELS_BASE_PATH,cfg['model_path']),custom_objects={"GroupNormalization":GroupNormalization} if cfg.get('has_custom_layer') else None,compile=False),
                "mean":np.load(os.path.join(MODELS_BASE_PATH,cfg['mean_path'])), "std":np.load(os.path.join(MODELS_BASE_PATH,cfg['std_path'])), **cfg}
            print(f"  ✅ Loaded: {cfg['name']}")
        except Exception as e: print(f"  ❌ FAILED to load '{cfg.get('name')}': {e}"); all_resources[cfg['name']]=None
    print("--- Model loading complete. ---\n")

    final_results = []
    for patient_id in patient_list:
        print("="*60 + f"\n--- Analyzing Patient ID: {patient_id} ---")
        patient_files = [os.path.join(audio_dir,f) for f in os.listdir(audio_dir) if f.startswith(str(patient_id)+'_') and f.endswith('.wav')]
        clf_resources = all_resources.get(CLASSIFIER_CONFIG['name'])
        classified_stage = get_classifier_prediction(patient_files,clf_resources)
        print(f"  -> Primary Classifier Verdict: {classified_stage}")
        prog_risk,check_label = "N/A","Progression Check"
        if classified_stage in EXPERT_CONFIGS:
            exp_config=EXPERT_CONFIGS[classified_stage];exp_res=all_resources.get(exp_config['name'])
            if exp_res:
                next_stage=exp_config['labels'][1];check_label=f"Risk of -> {next_stage}"
                prog_risk = get_progression_risk_percentage(patient_files,exp_res)
            else: prog_risk = "Expert Model Load Failed"
        elif classified_stage=="COPD4": prog_risk="Final Stage"
        print(f"  -> Progression Risk Assessment: {prog_risk}")
        final_results.append({"Patient ID":patient_id,"Classified Stage":classified_stage,"Progression Analysis":prog_risk})
    print("\n\n"+"="*90+"\n--- 📋 FINAL BATCH DIAGNOSTIC SUMMARY ---\n"+"="*90)
    print(pd.DataFrame(final_results).fillna("N/A").to_string(index=False))
    print("="*90)

if __name__ == "__main__":
    main(sorted(list(set(PATIENTS_TO_ANALYZE))), NEW_AUDIO_DIR)

--- Pre-loading all models... ---
  ✅ Loaded: Primary Classifier
  ✅ Loaded: Expert 0vs1
  ✅ Loaded: Expert 1vs2
  ✅ Loaded: Expert 2vs3
  ✅ Loaded: Expert 3vs4
--- Model loading complete. ---

--- Analyzing Patient ID: 104 ---
  -> Primary Classifier Verdict: COPD4
  -> Progression Risk Assessment: Final Stage
--- Analyzing Patient ID: 106 ---
  -> Primary Classifier Verdict: COPD3
  -> Progression Risk Assessment: 20.39%
--- Analyzing Patient ID: 107 ---
  -> Primary Classifier Verdict: COPD3
  -> Progression Risk Assessment: 6.37%
--- Analyzing Patient ID: 109 ---
  -> Primary Classifier Verdict: COPD4
  -> Progression Risk Assessment: Final Stage
--- Analyzing Patient ID: 110 ---
  -> Primary Classifier Verdict: COPD4
  -> Progression Risk Assessment: Final Stage
--- Analyzing Patient ID: 112 ---
  -> Primary Classifier Verdict: COPD4
  -> Progression Risk Assessment: Final Stage
--- Analyzing Patient ID: 113 ---
  -> Primary Classifier Verdict: COPD4
  -> Progression Risk Assessme